### Ingesting Results File

1) Read the file using spark dataframe reader API
2) Defined and enforced schema (preserved the nested structure)
3) Added metadata columns or audit columns such as:

    -- Source File

    --Ingestion Timestamp

4) Wrote the tables into to Bronze Layer    

Reading the CSV file

In [0]:
%run "../00-Common/01. Env Config"

In [0]:
%run "../00-Common/02. Bronze-helper_func"

In [0]:
# Defining relevant Source and Table name

Source_file=f"{landing_folder_path}/results" 
Table_name=f"{catalog_name}.{bronze_schema}.results"

This dataset is in nested json format. So first, nested json will be addressed and the schema will be dinfed for that followed by other objects.

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType, DateType, FloatType

results_Schema=StructType([
                          StructField("date",DateType()),
                          StructField("raceName",StringType()),
                          StructField("round",IntegerType()),
                          StructField("season",IntegerType()),
                          StructField("url",StringType()),
                          StructField("constructorId",StringType()),
                          StructField("driverId",StringType()),
                          StructField("grid",IntegerType()),
                          StructField("laps",IntegerType()),
                          StructField("number",IntegerType()),
                          StructField("points",FloatType()),
                          StructField("position",IntegerType()),
                          StructField("positionText",StringType()),
                          StructField("status",StringType()),
])


In [0]:
results_df=(
                spark.read.format('json')
                #.option('header',True)
                #.option('inferSchema','true')
                .option('mode','FAILFAST')      # PERMISSIVE, DROPMALFORMED, FAILFAST
                .schema(results_Schema)
                .load(Source_file))

In [0]:
display(results_df)

2) Adding Metadata

In [0]:
Results_final_df= add_ingestion_metadata(results_df)

In [0]:
display(Results_final_df)

3) Witting data to Bronze layer/delta table

In [0]:
(
    Results_final_df
    .write
    .format('delta')
    .mode('overwrite')
    .saveAsTable(Table_name)
)

In [0]:
display(spark.table(Table_name))